In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

conf = (pyspark.SparkConf()
    .setAppName('my-queries')
    .set('spark.jars.packages', 
         'org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,'
         'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,'
         'software.amazon.awssdk:bundle:2.17.178,'
         'software.amazon.awssdk:url-connection-client:2.17.178')
    .set('spark.sql.extensions', 
         'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,'
         'org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
    .set('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog')
    .set('spark.sql.catalog.nessie.uri', 'http://nessie:19120/api/v1')
    .set('spark.sql.catalog.nessie.ref', 'main')
    .set('spark.sql.catalog.nessie.authentication.type', 'NONE')
    .set('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
    .set('spark.sql.catalog.nessie.warehouse', 's3a://lakehouse/warehouse')
    .set('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .set('spark.sql.catalog.nessie.s3.endpoint', 'http://minio:9000')
    .set('spark.hadoop.fs.s3a.access.key', 'admin')
    .set('spark.hadoop.fs.s3a.secret.key', 'password123')
    .set('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .set('spark.hadoop.fs.s3a.path.style.access', 'true')
    .set('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .set('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem'))

spark = SparkSession.builder.config(conf=conf).getOrCreate()
print("✅ Spark ready!")

:: loading settings :: url = jar:file:/home/docker/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/docker/.ivy2/cache
The jars for the packages stored in: /home/docker/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.3_2.12 added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.3_2.12 added as a dependency
software.amazon.awssdk#bundle added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a95c46a1-cec6-4ff5-ad8c-5cc0113a1743;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.3_2.12;1.3.1 in central
	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.3_2.12;0.67.0 in central
	found software.amazon.awssdk#bundle;2.17.178 in central
	found software.amazon.eventstream#eventstream;1.0.1 in central
	found software.amazon.awssdk#url-connection-client;2.17.178 in central
	found software.amazon.awssdk#utils;2.17.178 in central
	found org.reactivestreams#reactive-streams;1.0.3 in central

25/12/21 07:24:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Spark ready!


In [2]:
spark.sql("SHOW TABLES IN nessie.ecommerce").show()


+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|ecommerce|customer_summary|      false|
+---------+----------------+-----------+



In [3]:
spark.sql("""
    SELECT 
        customer_id,
        name,
        total_orders,
        total_revenue,
        customer_segment
    FROM nessie.ecommerce.customer_summary
    ORDER BY total_revenue DESC
    LIMIT 10
""").show(truncate=False)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+-----------+------------+------------+-------------+----------------+
|customer_id|name        |total_orders|total_revenue|customer_segment|
+-----------+------------+------------+-------------+----------------+
|CUST0018   |Customer 18 |6           |4333.58      |Premium         |
|CUST0142   |Customer 142|4           |3015.39      |Premium         |
|CUST0064   |Customer 64 |4           |2974.08      |Premium         |
|CUST0098   |Customer 98 |4           |2626.4       |Premium         |
|CUST0076   |Customer 76 |4           |2574.76      |Premium         |
|CUST0150   |Customer 150|4           |2419.89      |Premium         |
|CUST0125   |Customer 125|3           |2152.29      |Premium         |
|CUST0155   |Customer 155|2           |1979.18      |Premium         |
|CUST0073   |Customer 73 |2           |1894.5       |Premium         |
|CUST0029   |Customer 29 |3           |1852.05      |Premium         |
+-----------+------------+------------+-------------+----------------+



In [4]:
spark.sql("""
    SELECT 
        customer_segment,
        COUNT(*) as customers,
        ROUND(SUM(total_revenue), 2) as total_revenue
    FROM nessie.ecommerce.customer_summary
    GROUP BY customer_segment
    ORDER BY total_revenue DESC
""").show()

+----------------+---------+-------------+
|customer_segment|customers|total_revenue|
+----------------+---------+-------------+
|         Premium|       44|     72592.67|
|            Gold|       62|     48245.73|
|          Silver|       37|     10934.07|
|          Bronze|        8|       516.99|
|       No Orders|       49|          0.0|
+----------------+---------+-------------+



In [6]:
# See raw data from bronze
spark.sql("SELECT * FROM nessie.ecommerce.`orders_bronze@bronze` LIMIT 5").show()



+---------+-----------+-------------------+------------+---------+--------------------+
| order_id|customer_id|         order_date|total_amount|   status|          created_at|
+---------+-----------+-------------------+------------+---------+--------------------+
|ORD000001|   CUST0164|2024-04-01 00:00:00|      139.74| refunded|2025-11-30 19:29:...|
|ORD000002|   CUST0029|2024-06-03 00:00:00|      382.95|cancelled|2025-11-30 19:29:...|
|ORD000003|   CUST0007|2024-01-08 00:00:00|      170.62| refunded|2025-11-30 19:29:...|
|ORD000004|   CUST0190|2024-12-28 00:00:00|      833.03|  pending|2025-11-30 19:29:...|
|ORD000005|   CUST0071|2024-09-30 00:00:00|      773.39|  pending|2025-11-30 19:29:...|
+---------+-----------+-------------------+------------+---------+--------------------+



In [7]:
# See cleaned data from silver  
spark.sql("SELECT * FROM nessie.ecommerce.`orders_silver@silver` LIMIT 5").show()

+---------+-----------+-------------------+------------+---------+--------------------+------------------+--------------------+-------------+
| order_id|customer_id|         order_date|total_amount|   status|          created_at|data_quality_score|        processed_at|source_branch|
+---------+-----------+-------------------+------------+---------+--------------------+------------------+--------------------+-------------+
|ORD000001|   CUST0164|2024-04-01 00:00:00|      139.74| refunded|2025-11-30 19:29:...|               100|2025-12-21 07:16:...|       bronze|
|ORD000002|   CUST0029|2024-06-03 00:00:00|      382.95|cancelled|2025-11-30 19:29:...|               100|2025-12-21 07:16:...|       bronze|
|ORD000003|   CUST0007|2024-01-08 00:00:00|      170.62| refunded|2025-11-30 19:29:...|               100|2025-12-21 07:16:...|       bronze|
|ORD000004|   CUST0190|2024-12-28 00:00:00|      833.03|  pending|2025-11-30 19:29:...|               100|2025-12-21 07:16:...|       bronze|
|ORD00